In [11]:
# ==============================
# REGRESSION MODEL (PRICE)
# ==============================

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import joblib
import os

# Define features and target
X_reg = df.drop(columns=["price"])
y_reg = df["price"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# Pipeline
reg_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", SGDRegressor(max_iter=1000, random_state=42))
])

# Train model
reg_pipeline.fit(X_train, y_train)

# Predictions
y_pred = reg_pipeline.predict(X_test)

# Evaluation
print("MSE:", mean_squared_error(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))

# Cross-validation
cv_scores = cross_val_score(
    reg_pipeline, X_reg, y_reg, cv=5, scoring="neg_mean_squared_error"
)

print("Cross Validation MSE:", -cv_scores.mean())

# Save model (FIXED)
os.makedirs("../artifacts", exist_ok=True)
joblib.dump(reg_pipeline, "../artifacts/reg_model.joblib")

print("Regression model saved successfully!")

2026/04/04 20:26:35 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'dfb583d0e8c740a087a8c19c0122bdc8', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/04/04 20:26:35 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\amils\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Inte

MSE: 20665.523521865627
MAE: 123.99309659200559
Cross Validation MSE: 20071.64132555682
Regression model saved successfully!


In [12]:
# ==============================
# CLASSIFICATION MODEL (CUSTOMER SEGMENT)
# ==============================

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import mlflow
import mlflow.sklearn
import joblib
import os

# Define features and target
X_cls = df.drop("customer_segment", axis=1)
y_cls = df["customer_segment"]

# Split data (IMPORTANT: stratify)
X_train, X_test, y_train, y_test = train_test_split(
    X_cls, y_cls, test_size=0.2, stratify=y_cls, random_state=42
)

# Pipeline
clf_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(
        max_iter=500,
        class_weight="balanced",
        solver="lbfgs"
    ))
])

# Create artifacts folder if not exists
os.makedirs("../artifacts", exist_ok=True)

# MLflow tracking
mlflow.set_experiment("Ecommerce_Project")
mlflow.sklearn.autolog()

with mlflow.start_run():
    # Train model
    clf_pipeline.fit(X_train, y_train)

    # Predictions
    y_pred = clf_pipeline.predict(X_test)

    # Evaluation
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    # Probabilities (for analysis)
    probs = clf_pipeline.predict_proba(X_test)
    print("\nSample Probabilities:")
    print(probs[:5])

    # Save model (IMPORTANT)
    joblib.dump(clf_pipeline, "../artifacts/model.joblib")
    print("\nModel saved successfully!")

2026/04/04 20:26:46 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\amils\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/04/04 20:26:46 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\amils\AppData\Lo

Confusion Matrix:
[[ 32  35  46]
 [328 318 211]
 [392 396 242]]

Classification Report:
              precision    recall  f1-score   support

         New       0.04      0.28      0.07       113
   Returning       0.42      0.37      0.40       857
         VIP       0.48      0.23      0.32      1030

    accuracy                           0.30      2000
   macro avg       0.32      0.30      0.26      2000
weighted avg       0.43      0.30      0.34      2000


Sample Probabilities:
[[0.34466751 0.30826646 0.34706603]
 [0.40098857 0.29875497 0.30025646]
 [0.33119992 0.34855359 0.3202465 ]
 [0.35110568 0.31235348 0.33654084]
 [0.272195   0.38772696 0.34007804]]

Model saved successfully!
